In [73]:
%run common_setup.ipynb

In [74]:
class ETL(SetUp):

    def __init__(self):
        super().__init__()
        self.assemble_tables = {}
        return
    
    def match_table(self):
        sql = """  

            -- MATCH Domingo's EconBus list to OpenAlex
            -- ========================================
            WITH
            authorships_CTE AS
                (SELECT DISTINCT -- work_id,
                    works_count,
                    cited_by_count,
                    h_index,
                    sub.author_name,
                    sub.author_id,
                    display_name_alternatives,
                    list(institution.display_name) AS institution_name,
                    list(institution.id) AS institution_id,
                    list(institution.country_code) AS country_code,
                    concat(first[1],'. ', last)
                FROM
                    (SELECT work_id,
                        authorship.author.display_name as author_name,
                        authorship.author.id AS author_id,
                        unnest(authorship.institutions) AS institution,
                    FROM 
                        (SELECT id AS work_id,
                            unnest(authorships) AS authorship
                        FROM project.raw
                        )
                    ) sub
                LEFT JOIN project.authors_full
                USING (author_id)
                GROUP BY ALL
                )
            SELECT *
            FROM project.domingo_sample_original d
            LEFT JOIN authorships_CTE a
            ON list_contains(display_name_alternatives, fullname) OR list_contains(display_name_alternatives, concat(first[1],'. ', last))
            WHERE author_id IS NULL
        """
        sample = self.db.sql(sql).df()
        sample = sample.sort_values(['Group', 'Research_Profile'], ascending=[False, True]).reset_index(drop=True)
        print(f'{sample.shape = }\n{sample.head()}')
        self.assemble_tables |= {'Domingo_matched': sample}
        return
    
    def works_table(self):
        sql = """
            -- ETL works for Domingo
            -- =====================
            SELECT id as work_id,
                    doi, 
                    title, 
                    publication_year,
                    fwci,
                    cited_by_count,
                    referenced_works_count,
                    "primary_location.source".display_name AS source_name,
                    "primary_location.source".issn_l AS issn,
                    "primary_location.source".host_organization_name AS publisher,
                    "biblio.volume" AS volume,
                FROM project.raw
            """
        works = self.db.sql(sql).df()
        works = works.sort_values(['publication_year', 'fwci'], ascending=[True, False]).reset_index(drop=True)
        print(f'{works.shape = }\n{works.head()}')
        self.assemble_tables |= {'works': works}
        return
    
    def authorships_table(self):
        sql = """
            -- ETL FOR Domingo authors and institutions
            -- ========================================
            SELECT work_id,
                    author_name,
                    author_id,
                    institution.display_name AS institution_name,
                    institution.id AS institution_id,
                    institution.country_code AS country_code
            FROM
                (SELECT work_id,
                    authorship.author.display_name as author_name,
                    authorship.author.id AS author_id,
                    unnest(authorship.institutions) AS institution,
                FROM 
                    (SELECT id AS work_id,
                        unnest(authorships) AS authorship
                    FROM project.raw
                    )
                )
                """
        authorships = self.db.sql(sql).df()
        print(f"{authorships.shape = }\n{authorships.head()}")
        self.assemble_tables |= {'authorships': authorships}
        return
    
    def references_table(self):
        sql = """
            -- ETL FOR Domingo reference_list
            -- ==============================
            SELECT id AS work_id,
                    referenced_works
                FROM project.raw
            """
        references = self.db.sql(sql).df()
        print(f'{references.shape = }\n{references.head()}')
        self.assemble_tables |= {'references': references}
        return
    
    def citations_table(self):
        sql = """
            -- ETL FOR Domingo citation_list
            -- ==============================
            SELECT cited_id,
                    list(citer_id) AS citing_works_list
            FROM 
                (SELECT id AS citer_id,
                    unnest(referenced_works) AS cited_id
                FROM project.raw
                )
            GROUP BY ALL
            """
        citations = self.db.sql(sql).df()
        print(f'{citations.shape = }\n{citations.head()}')
        self.assemble_tables |= {'citations': citations}
        return
    
    def topics_table(self):
        sql = """ 
            -- ETL FOR Domingo topics
            -- ======================
            SELECT id AS work_id,
                    "primary_topic.display_name" AS topic_name,
                    "primary_topic.subfield".display_name AS subfield_name,
                    "primary_topic.field".display_name AS field_name,
                    "primary_topic.domain".display_name AS domain_name,
                    "primary_topic.score" AS topic_score
            FROM project.raw
            """
        topics = self.db.sql(sql).df()
        print(f'{topics.shape = }\n{topics.head()}')
        self.assemble_tables |= {'topics': topics}
        return


    def load_tables(self):
        for k, v in self.assemble_tables.items():
            with open(f'../DATA/tables_for_Domingo_{k}.csv', 'w') as writer:
                print(f'{k = } {v.shape = }\n{v.head()}')
                v.to_csv(writer, index=False)


In [75]:
def main():

    etl = ETL()
    etl.match_table()
    etl.works_table()
    etl.authorships_table()
    etl.references_table()
    etl.citations_table()
    etl.topics_table()
    etl.load_tables()
    etl.db.close()

    return

In [76]:
if __name__ == "__main__":
    main()
    print("DONE")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ backup   │ main    │ authors_full         │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR]  │ false     │
│ backup   │ main    │ citer_cited          │ [citer_id, citer_y…  │ [VARCHAR, BIGINT, VARCHAR, BIGINT, …  │ false     │
│ backup   │ main    │ raw                  │ [id, doi, title, p…  │ [VARCHAR, VARCHAR, VARCHAR, BIGINT,…  │ false     │
│ backup   │ main    │ sources_o